In [ ]:
!pip install -q transformers datasets torch scikit-learn tqdm

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1 — Config
# ════════════════════════════════════════════════════════════════

import torch
import transformers
transformers.logging.set_verbosity_error()   # suppress the >512 token warnings

CHUNK_SIZE  = 512          # BERT's hard limit
STRIDE      = 256          # step between chunks → 256 token overlap
MAX_CHUNKS  = 8            # cap per document to avoid OOM
GRU_HIDDEN  = 256          # hidden size per direction
GRU_LAYERS  = 2
DROPOUT     = 0.3
LR          = 3e-4         # GRU + classifier only (BERT is frozen)
BATCH_SIZE  = 4
EPOCHS      = 5
MODEL_NAME  = "law-ai/InLegalBERT"
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2 — Dataset
# ════════════════════════════════════════════════════════════════

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ILDCHierarchicalDataset(Dataset):

    def __init__(self, df, tokenizer, chunk_size=512, stride=256, max_chunks=20):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.chunk_size = chunk_size
        self.stride     = stride
        self.max_chunks = max_chunks

    def __len__(self):
        return len(self.df)

    def _chunk_text(self, text):
        # Tokenize full document — no truncation, no special tokens yet
        encoding = self.tokenizer(
            str(text),
            add_special_tokens=False,
            return_tensors=None
        )
        all_ids  = encoding['input_ids']
        all_mask = encoding['attention_mask']

        # Each chunk: [CLS] + 510 tokens + [SEP] = 512
        effective = self.chunk_size - 2
        chunks    = []
        start     = 0

        while start < len(all_ids):
            end        = start + effective
            chunk_ids  = [tokenizer.cls_token_id] + all_ids[start:end]  + [tokenizer.sep_token_id]
            chunk_mask = [1]                       + all_mask[start:end] + [1]

            # Pad to exactly 512
            pad        = self.chunk_size - len(chunk_ids)
            chunk_ids  += [tokenizer.pad_token_id] * pad
            chunk_mask += [0]                      * pad

            chunks.append({
                'input_ids':      torch.tensor(chunk_ids,  dtype=torch.long),
                'attention_mask': torch.tensor(chunk_mask, dtype=torch.long)
            })

            if end >= len(all_ids):
                break
            start += self.stride          # stride=256 → 256 token overlap

        return chunks[:self.max_chunks]   # hard cap

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        chunks = self._chunk_text(row['text'])
        label  = torch.tensor(int(row['label']), dtype=torch.long)
        return chunks, label


def collate_fn(batch):
    """
    Pads all documents in the batch to the same number of chunks.
    Returns tensors of shape (B, max_chunks, 512).
    """
    chunk_lists, labels = zip(*batch)
    max_chunks = max(len(c) for c in chunk_lists)
    chunk_size = chunk_lists[0][0]['input_ids'].shape[0]

    batch_ids, batch_masks, chunk_counts = [], [], []

    for chunks in chunk_lists:
        n = len(chunks)
        chunk_counts.append(n)

        ids   = torch.stack([c['input_ids']      for c in chunks])  # (n, 512)
        masks = torch.stack([c['attention_mask']  for c in chunks])  # (n, 512)

        # Pad chunk dimension with zeros if this doc has fewer chunks
        pad = max_chunks - n
        if pad > 0:
            ids   = torch.cat([ids,   torch.zeros(pad, chunk_size, dtype=torch.long)])
            masks = torch.cat([masks, torch.zeros(pad, chunk_size, dtype=torch.long)])

        batch_ids.append(ids)
        batch_masks.append(masks)

    return {
        'input_ids':      torch.stack(batch_ids),            # (B, max_chunks, 512)
        'attention_mask': torch.stack(batch_masks),           # (B, max_chunks, 512)
        'chunk_counts':   torch.tensor(chunk_counts),         # (B,)
        'labels':         torch.stack(labels)                 # (B,)
    }

config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3 — Model
# ════════════════════════════════════════════════════════════════

import torch.nn as nn
from transformers import AutoModel


class HierarchicalLegalBERT(nn.Module):

    def __init__(self, model_name, gru_hidden=256, gru_layers=2,
                 dropout=0.3, num_classes=2):
        super().__init__()

        # ── InLegalBERT (frozen) ──────────────────────────────
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False          # freeze all BERT weights

        bert_dim = self.bert.config.hidden_size  # 768

        # ── GRU over chunk [CLS] embeddings ──────────────────
        # Bidirectional → final hidden is gru_hidden * 2
        self.gru = nn.GRU(
            input_size    = bert_dim,
            hidden_size   = gru_hidden,
            num_layers    = gru_layers,
            batch_first   = True,
            dropout       = dropout if gru_layers > 1 else 0,
            bidirectional = True
        )

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(gru_hidden * 2, num_classes)

    def forward(self, input_ids, attention_mask, chunk_counts):
        """
        input_ids      : (B, max_chunks, 512)
        attention_mask : (B, max_chunks, 512)
        chunk_counts   : (B,) — real chunks per doc, rest is padding
        """
        B, max_chunks, seq_len = input_ids.shape

        # Flatten → (B * max_chunks, 512) so BERT can process all chunks at once
        ids   = input_ids.view(B * max_chunks, seq_len)
        masks = attention_mask.view(B * max_chunks, seq_len)

        # BERT forward — no grad since frozen
        with torch.no_grad():
            bert_out = self.bert(input_ids=ids, attention_mask=masks)

        # [CLS] token embedding for each chunk → (B * max_chunks, 768)
        cls_emb = bert_out.last_hidden_state[:, 0, :]

        # Reshape back to (B, max_chunks, 768)
        cls_emb = cls_emb.view(B, max_chunks, -1)
        cls_emb = self.dropout(cls_emb)

        # Pack padded sequence — GRU ignores the zero-padded chunk slots
        packed = nn.utils.rnn.pack_padded_sequence(
            cls_emb,
            chunk_counts.cpu(),
            batch_first    = True,
            enforce_sorted = False
        )
        _, hidden = self.gru(packed)
        # hidden: (num_layers * 2, B, gru_hidden) — 2 because bidirectional

        # Last layer: forward direction = hidden[-2], backward = hidden[-1]
        final = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (B, gru_hidden*2)
        final = self.dropout(final)

        return self.classifier(final)                        # (B, num_classes)

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4 — Train / Eval loops
# ════════════════════════════════════════════════════════════════

from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in tqdm(loader, desc="  Train"):
        ids    = batch['input_ids'].to(device)
        masks  = batch['attention_mask'].to(device)
        counts = batch['chunk_counts']             # stays on CPU for pack_padded
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(ids, masks, counts)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        all_preds   += logits.argmax(dim=-1).cpu().tolist()
        all_labels  += labels.cpu().tolist()

    return {
        'loss': total_loss / len(loader),
        'acc':  accuracy_score(all_labels, all_preds),
        'f1':   f1_score(all_labels, all_preds, average='macro')
    }


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in tqdm(loader, desc="  Eval "):
        ids    = batch['input_ids'].to(device)
        masks  = batch['attention_mask'].to(device)
        counts = batch['chunk_counts']
        labels = batch['labels'].to(device)

        logits = model(ids, masks, counts)
        loss   = criterion(logits, labels)

        total_loss  += loss.item()
        all_preds   += logits.argmax(dim=-1).cpu().tolist()
        all_labels  += labels.cpu().tolist()

    return {
        'loss': total_loss / len(loader),
        'acc':  accuracy_score(all_labels, all_preds),
        'f1':   f1_score(all_labels, all_preds, average='macro')
    }

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5 — Run
# ════════════════════════════════════════════════════════════════

import pandas as pd
from torch.optim import AdamW
from datasets import load_dataset


def apply_criminal_filter(df, verbose=False):
    # Drop expert cols, nulls, duplicates
    df = df[['id', 'text', 'label']].dropna().drop_duplicates(subset=['text']).reset_index(drop=True)

    tier1 = df['text'].str.contains(
        r'Criminal Appeal|Criminal Writ Petition|Crl\.?\s*A\b',
        case=False, na=False, regex=True)

    tier2 = df['text'].str.contains(
        r'\bSessions\s+(Case|Judge|Court)\b'
        r'|\bSection\s+30[234567]\b'
        r'|\bSection\s+376\b'
        r'|\bdeath\s+sentence\b'
        r'|\blife\s+imprisonment\b'
        r'|\bacquitted\b'
        r'|\bcharge\s*sheet\b',
        case=False, na=False, regex=True)

    exclude = df['text'].str.contains(
        r'compulsory retirement|service matter'
        r'|disciplinary proceed|departmental proceed'
        r'|food adulteration|Prevention of Food'
        r'|Article\s+105|Article\s+226',
        case=False, na=False, regex=True)

    filtered = df[(tier1 | tier2) & ~exclude].copy().reset_index(drop=True)

    if verbose:
        print(f"Total                 : {len(df):,}")
        print(f"Tier 1 (header)       : {tier1.sum():,}")
        print(f"Tier 2 (strong terms) : {tier2.sum():,}")
        print(f"Excluded              : {((tier1 | tier2) & exclude).sum():,}")
        print(f"Final criminal subset : {len(filtered):,} ({len(filtered)/len(df)*100:.1f}%)")
        print(f"  Rejected (0) : {(filtered['label']==0).sum():,} ({(filtered['label']==0).mean()*100:.1f}%)")
        print(f"  Accepted (1) : {(filtered['label']==1).sum():,} ({(filtered['label']==1).mean()*100:.1f}%)")
        print("\n--- Spot check (5 random cases) ---")
        for _, row in filtered.sample(5, random_state=42).iterrows():
            print(f"\nID: {row['id']} | Label: {row['label']}")
            print(f"{str(row['text'])[:300]}")
            print("-"*60)

    return filtered


def run():

    # ── Load + filter ─────────────────────────────────────────
    print("Loading multi_train...")
    train_df = load_dataset("Exploration-Lab/IL-TUR", "cjpe", split="multi_train").to_pandas()
    train_df = apply_criminal_filter(train_df, verbose=True)

    print("\nLoading multi_dev...")
    dev_df = load_dataset("Exploration-Lab/IL-TUR", "cjpe", split="multi_dev").to_pandas()
    dev_df = apply_criminal_filter(dev_df, verbose=True)

    print(f"\nTrain : {len(train_df):,}  (0: {(train_df['label']==0).sum():,} | 1: {(train_df['label']==1).sum():,})")
    print(f"Dev   : {len(dev_df):,}   (0: {(dev_df['label']==0).sum():,}  | 1: {(dev_df['label']==1).sum():,})")

    # ── Datasets & loaders ───────────────────────────────────
    train_ds = ILDCHierarchicalDataset(train_df, tokenizer, CHUNK_SIZE, STRIDE, MAX_CHUNKS)
    dev_ds   = ILDCHierarchicalDataset(dev_df,   tokenizer, CHUNK_SIZE, STRIDE, MAX_CHUNKS)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  collate_fn=collate_fn, num_workers=2)
    dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE,
                              shuffle=False, collate_fn=collate_fn, num_workers=2)

    # ── Model ─────────────────────────────────────────────────
    model = HierarchicalLegalBERT(MODEL_NAME, GRU_HIDDEN, GRU_LAYERS, DROPOUT).to(DEVICE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"\nTrainable params : {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

    # ── Loss — weighted for label imbalance ──────────────────
    counts    = train_df['label'].value_counts().sort_index().values
    weights   = torch.tensor(1.0 / counts, dtype=torch.float)
    weights   = (weights / weights.sum()).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # ── Optimizer — only non-frozen params ───────────────────
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=1e-2
    )

    # ── Training loop ─────────────────────────────────────────
    best_f1 = 0.0
    history = []

    for epoch in range(1, EPOCHS + 1):
        print(f"\n{'='*55}")
        print(f"  Epoch {epoch}/{EPOCHS}")
        print(f"{'='*55}")

        tr = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        dv = eval_epoch(model,  dev_loader,   criterion,            DEVICE)

        history.append({'epoch': epoch, **{f'train_{k}': v for k, v in tr.items()},
                                         **{f'dev_{k}':   v for k, v in dv.items()}})

        print(f"  Train — loss: {tr['loss']:.4f} | acc: {tr['acc']:.4f} | macro-F1: {tr['f1']:.4f}")
        print(f"  Dev   — loss: {dv['loss']:.4f} | acc: {dv['acc']:.4f} | macro-F1: {dv['f1']:.4f}")

        if dv['f1'] > best_f1:
            best_f1 = dv['f1']
            torch.save(model.state_dict(), 'best_hierarchical_model.pt')
            print(f"  ✓ Best model saved  (macro-F1: {best_f1:.4f})")

    print(f"\nDone. Best dev macro-F1: {best_f1:.4f}")
    pd.DataFrame(history).to_csv('training_history.csv', index=False)
    return model


model = run()

Loading multi_train...


README.md: 0.00B [00:00, ?B/s]

cjpe/expert-00000-of-00001.parquet:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

cjpe/single_train-00000-of-00001.parquet:   0%|          | 0.00/56.4M [00:00<?, ?B/s]

cjpe/single_dev-00000-of-00001.parquet:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

cjpe/multi_train-00000-of-00002.parquet:   0%|          | 0.00/234M [00:00<?, ?B/s]

... (24 outputs left out)

  Eval : 100%|██████████| 39/39 [00:45<00:00,  1.16s/it]


  Train — loss: 0.6823 | acc: 0.5767 | macro-F1: 0.5632
  Dev   — loss: 0.6926 | acc: 0.5260 | macro-F1: 0.5109

  Epoch 5/5


  Eval : 100%|██████████| 39/39 [00:45<00:00,  1.16s/it]

  Train — loss: 0.6783 | acc: 0.5796 | macro-F1: 0.5689
  Dev   — loss: 0.6983 | acc: 0.5130 | macro-F1: 0.4975

Done. Best dev macro-F1: 0.5475


In [ ]:
from datasets import load_dataset

train_raw = load_dataset("Exploration-Lab/IL-TUR", "cjpe", split="multi_train").to_pandas()
dev_raw   = load_dataset("Exploration-Lab/IL-TUR", "cjpe", split="multi_dev").to_pandas()

train_df = apply_criminal_filter(train_raw)
dev_df   = apply_criminal_filter(dev_raw)

train_df.to_parquet("criminal_train.parquet", index=False)
dev_df.to_parquet("criminal_dev.parquet", index=False)

print(f"Saved: {len(train_df):,} train | {len(dev_df):,} dev rows")

/tmp/ipykernel_3120/1695377556.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tier2 = df['text'].str.contains(
/tmp/ipykernel_3120/1695377556.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  tier2 = df['text'].str.contains(


Saved: 7,094 train | 154 dev rows


In [ ]:
import pandas as pd

df = pd.read_parquet("criminal_train.parquet")

# Basic info
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nNull values:\n{df.isnull().sum()}")

# Sample 5 cases and print first 500 chars of each
print("\n--- Sample Cases ---")
for _, row in df.sample(5, random_state=42).iterrows():
    print(f"\nID: {row['id']} | Label: {row['label']}")
    print(row['text'][:500])
    print("-" * 60)

Shape: (7094, 3)

Columns: ['id', 'text', 'label']

Label distribution:
label
0    4095
1    2999
Name: count, dtype: int64

Null values:
id       0
text     0
label    0
dtype: int64

--- Sample Cases ---

ID: 1999_925 | Label: 1
 PATTANAIK,J. The State of Kerala is in appeal against the Judgment dated 4.2.94 of the Kerala High Court in Criminal Revision Petition No. 521 of 1988. By the impugned Judgment, the High Court in revision, has interfered with the companyviction and sentence passed against the accused respondent of the offences under Sections 408, 468 and 477A of the Indian Penal Code. The accused-respondent was an employee of Western India Plywoods and was head of the purchase section. In companyrse of his duti
------------------------------------------------------------

ID: 1984_336 | Label: 0
CRIMINAL APPELLATE JURISDICTION Criminal Appeal Nos. 461-466 of 1980. From the Judgment order dated 9.11.78 of the Allahabad High Court in Criminal Misc. Application Nos. 1655, 1656,